# Preprocessing demo

I check how messy URL strings are cleaned and turned into model features.

In [1]:
from pathlib import Path
import os
import sys

# Keep imports working when nbconvert runs from notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from preprocess import (STRUCTURE_NAMES, advanced_features, baseline_features,
                        clean_url, eval_key, registered_domain, split_url,
                        structure)

In [2]:
# These examples cover common formatting noise without changing the feature rules.
examples = [
    "HTTPS://WWW.PayPal.com/SignIn/",
    "http://example.com/a&amp;b",
    None,
    "http://bad.com/\x00login\n",
    "https://例え.テスト/パス",
]
display(pd.DataFrame([{
    "raw": str(raw), "clean": clean_url(raw), "eval_key": eval_key(raw),
    "registered_domain": registered_domain(raw),
} for raw in examples]))

url = clean_url("https://www.PayPal-secure.login.tk/webscr?cmd=x")
host, path = split_url(url)
display(pd.DataFrame({"part": ["clean URL", "host", "path/query"],
                      "value": [url, host, path]}))

,raw,clean,eval_key,registered_domain
0,HTTPS://WWW.PayPal.com/SignIn/,https://www.paypal.com/signin/,paypal.com/signin,paypal.com
1,http://example.com/a&amp;b,http://example.com/a&b,example.com/a&b,example.com
2,None,,,
3,http://bad.com/ login\n,http://bad.com/login,bad.com/login,bad.com
4,https://例え.テスト/パス,https://例え.テスト/パス,例え.テスト/パス,例え.テスト


,part,value
0,clean URL,https://www.paypal-secure.login.tk/webscr?cmd=x
1,host,www.paypal-secure.login.tk
2,path/query,/webscr?cmd=x


In [3]:
# Hashing needs no vocabulary file, and sparse matrices stay manageable.
sample_urls = [url, "https://www.example.com/", "http://10.0.0.1/login"]
baseline = baseline_features(sample_urls)
advanced = advanced_features(sample_urls)
display(pd.DataFrame([
    {"features": "baseline", "shape": baseline.shape, "nonzero_values": baseline.nnz},
    {"features": "advanced", "shape": advanced.shape, "nonzero_values": advanced.nnz},
]))
display(pd.DataFrame({"feature": STRUCTURE_NAMES,
                      "value": structure(url, host, path)}))

,features,shape,nonzero_values
0,baseline,"(3, 65536)",246
1,advanced,"(3, 81942)",247


,feature,value
0,log_len_url,3.871201
1,log_len_host,3.295837
2,log_len_path,2.639057
3,log_host_labels,1.609438
4,log_host_hyphens,0.693147
5,log_host_digits,0.000000
6,log_path_slashes,0.693147
7,log_path_equals,0.693147
8,log_url_pct,0.000000
9,log_url_dots,1.386294
